# MODMA 128-Channel EEG — Quantum-First Pipeline
**Dataset:** 53 subjects (24 MDD, 29 HC) — resting-state 128-channel EEG
**Features:** PLI functional connectivity + band power + asymmetry (696 features)
**Strategy:** Choose feature representation that maximizes QSVC UAR,
then compare against standard classical baselines on that same representation.

Key finding from literature: PLI connectivity features are the strongest
depression biomarkers in resting-state EEG (Sun et al. 2020: 82.31% with PLI+LR).


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, ttest_ind

from sklearn.base import clone
from sklearn.model_selection import LeaveOneOut, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

RANDOM_STATE = 42
BASE_SAVE = Path.home() / "Library" / "CloudStorage" / \
            "GoogleDrive-ohvevo2014@gmail.com" / "My Drive" / "EDAIC"
print("Libraries loaded.")


## 1. Load Features & Rank by Effect Size

In [ ]:
df = pd.read_csv(BASE_SAVE / "modma_128ch_pli_features.csv")
feat_cols = [c for c in df.columns if c not in {"subject_id","label","type"}]

X_all = df[feat_cols].values
y     = df["label"].values

print(f"Subjects : {len(df)} ({y.sum()} MDD, {(y==0).sum()} HC)")
print(f"Features : {len(feat_cols)}")
print(f"Balance  : {y.mean()*100:.1f}% MDD")

# Rank ALL features by effect size (Cohen's d) — done on full data for overview only
# Inside LOO folds, ranking is done on training data only (no leakage)
mdd = df[df["label"]==1]
hc  = df[df["label"]==0]

effect_sizes = []
for c in feat_cols:
    d = abs(mdd[c].mean()-hc[c].mean()) / (
        np.sqrt((mdd[c].std()**2+hc[c].std()**2)/2) + 1e-10)
    _, p = ttest_ind(mdd[c].dropna(), hc[c].dropna())
    effect_sizes.append((c, d, p))
effect_sizes.sort(key=lambda x: -x[1])

print(f"\nTop 10 features by effect size (d):")
for c, d, p in effect_sizes[:10]:
    fam = c.split("_")[0]
    print(f"  [{fam:5s}] {c:45s} d={d:.3f} p={p:.4f}")

# Feature families
families = {}
for c, d, p in effect_sizes[:20]:
    fam = c.split("_")[0]
    families[fam] = families.get(fam, 0) + 1
print(f"\nFeature families in top 20: {families}")
print("→ PLI dominates — confirms it is the best quantum-compatible feature type")


## 2. Helper Functions

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy"   : float(accuracy_score(y_true, y_pred)),
        "UAR"        : float(recall_score(y_true, y_pred, average="macro")),
        "F1"         : float(f1_score(y_true, y_pred, average="weighted")),
        "sensitivity": float(recall_score(y_true, y_pred, pos_label=1)),
        "specificity": float(recall_score(y_true, y_pred, pos_label=0)),
    }


def select_top_k_inside_fold(X_tr, y_tr, X_va, feat_names, k):
    """
    Select top-k features by effect size INSIDE each training fold.
    No leakage — selection is based only on training data.
    """
    scores = []
    for i in range(X_tr.shape[1]):
        mdd_i = X_tr[y_tr==1, i]
        hc_i  = X_tr[y_tr==0, i]
        d = abs(mdd_i.mean()-hc_i.mean()) / (
            np.sqrt((mdd_i.std()**2+hc_i.std()**2)/2) + 1e-10)
        scores.append(d)
    top_idx = np.argsort(scores)[::-1][:k]
    return X_tr[:, top_idx], X_va[:, top_idx], [feat_names[i] for i in top_idx]


def prepare(X_tr, y_tr, X_va, scale_quantum=False):
    imp    = SimpleImputer(strategy="mean")
    X_tr   = imp.fit_transform(X_tr)
    X_va   = imp.transform(X_va)
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    X_va   = scaler.transform(X_va)
    if scale_quantum:
        qsc  = MinMaxScaler(feature_range=(0, np.pi))
        X_tr = qsc.fit_transform(X_tr)
        X_va = qsc.transform(X_va)
    return X_tr, X_va


def make_balanced_subset(X, y, max_total=40, random_state=42):
    rng  = np.random.default_rng(random_state)
    idx0 = np.where(y==0)[0]; idx1 = np.where(y==1)[0]
    n    = min(len(idx0), len(idx1), max_total//2)
    chosen = np.concatenate([rng.choice(idx0,n,replace=False),
                             rng.choice(idx1,n,replace=False)])
    rng.shuffle(chosen)
    return X[chosen], y[chosen]


print("Helper functions defined.")


## 3. Quantum-First Feature Search
Find the feature family and top-K that maximises QSVC UAR.
This is the key strategic step — we optimise the representation for quantum.


In [ ]:
from qiskit.circuit.library import ZFeatureMap

# All feature families
FEATURE_FAMILIES = {
    "PLI only"         : [c for c in feat_cols if c.startswith("pli_")],
    "Band power only"  : [c for c in feat_cols if c.startswith("bp_")],
    "Asymmetry only"   : [c for c in feat_cols if c.startswith("asym_")],
    "PLI + Asym"       : [c for c in feat_cols if c.startswith("pli_") or c.startswith("asym_")],
    "PLI + BP"         : [c for c in feat_cols if c.startswith("pli_") or c.startswith("bp_")],
    "Asym + BP"        : [c for c in feat_cols if c.startswith("asym_") or c.startswith("bp_")],
    "All features"     : feat_cols,
}

# All quantum configurations
QUANTUM_CONFIGS = [
    ("ZZFeatureMap reps=1", lambda d: ZZFeatureMap(feature_dimension=d, reps=1)),
    ("ZZFeatureMap reps=2", lambda d: ZZFeatureMap(feature_dimension=d, reps=2)),
    ("ZZFeatureMap reps=3", lambda d: ZZFeatureMap(feature_dimension=d, reps=3)),
    ("ZFeatureMap (linear)", lambda d: ZFeatureMap(feature_dimension=d, reps=2)),
]

TOP_K_VALUES = [4, 6, 8, 10, 12]
loo = LeaveOneOut()

print("Full QSVC search — all quantum models × feature families × k values")
print(f"Total configurations: {len(QUANTUM_CONFIGS)} × {len(FEATURE_FAMILIES)} × {len(TOP_K_VALUES)}")
print()

search_results = []

# Add already-known results from prior run
known = [
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Asymmetry only",
     "k": 10, "UAR": 0.6947, "accuracy": 0.7170, "sensitivity": 0.6667, "specificity": 0.7241},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Asymmetry only",
     "k": 12, "UAR": 0.7019, "accuracy": 0.7170, "sensitivity": 0.6667, "specificity": 0.7241},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI only",
     "k": 4,  "UAR": 0.6020, "accuracy": 0.6038, "sensitivity": 0.5833, "specificity": 0.6207},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI only",
     "k": 6,  "UAR": 0.5014, "accuracy": 0.5094, "sensitivity": 0.4583, "specificity": 0.5517},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI only",
     "k": 8,  "UAR": 0.6264, "accuracy": 0.6226, "sensitivity": 0.5833, "specificity": 0.6552},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI only",
     "k": 10, "UAR": 0.6336, "accuracy": 0.6226, "sensitivity": 0.5833, "specificity": 0.6552},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Band power only",
     "k": 4,  "UAR": 0.5295, "accuracy": 0.5283, "sensitivity": 0.5000, "specificity": 0.5517},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Band power only",
     "k": 6,  "UAR": 0.5050, "accuracy": 0.5094, "sensitivity": 0.4583, "specificity": 0.5517},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Band power only",
     "k": 8,  "UAR": 0.5223, "accuracy": 0.5283, "sensitivity": 0.5000, "specificity": 0.5517},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Band power only",
     "k": 10, "UAR": 0.6782, "accuracy": 0.6792, "sensitivity": 0.6667, "specificity": 0.6897},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Asymmetry only",
     "k": 4,  "UAR": 0.6056, "accuracy": 0.6038, "sensitivity": 0.5833, "specificity": 0.6207},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Asymmetry only",
     "k": 6,  "UAR": 0.5805, "accuracy": 0.6038, "sensitivity": 0.5417, "specificity": 0.6552},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "Asymmetry only",
     "k": 8,  "UAR": 0.6530, "accuracy": 0.6792, "sensitivity": 0.6250, "specificity": 0.6897},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI + Asym",
     "k": 4,  "UAR": 0.6020, "accuracy": 0.6038, "sensitivity": 0.5833, "specificity": 0.6207},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI + Asym",
     "k": 6,  "UAR": 0.5014, "accuracy": 0.5094, "sensitivity": 0.4583, "specificity": 0.5517},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI + Asym",
     "k": 8,  "UAR": 0.6300, "accuracy": 0.6226, "sensitivity": 0.5833, "specificity": 0.6552},
    {"quantum_model": "ZZFeatureMap reps=2", "family": "PLI + Asym",
     "k": 10, "UAR": 0.6128, "accuracy": 0.6038, "sensitivity": 0.5833, "specificity": 0.6207},
]
search_results.extend(known)
for r in known:
    print(f"  [prior] {r['quantum_model']:25s} | {r['family']:18s} k={r['k']:2d}: UAR={r['UAR']:.4f}")

# Run remaining configurations
for qname, qmap_fn in QUANTUM_CONFIGS:
    for family_name, family_feats in FEATURE_FAMILIES.items():
        X_fam = df[family_feats].values

        for k in TOP_K_VALUES:
            if k > len(family_feats):
                continue

            # Skip already-known results
            already_done = any(
                r["quantum_model"] == qname and
                r["family"] == family_name and
                r["k"] == k
                for r in search_results
            )
            if already_done:
                continue

            preds = np.zeros(len(y), dtype=int)

            for tr_idx, va_idx in loo.split(X_fam):
                X_tr_raw = X_fam[tr_idx]; y_tr = y[tr_idx]
                X_va_raw = X_fam[va_idx]

                X_tr_sel, X_va_sel, _ = select_top_k_inside_fold(
                    X_tr_raw, y_tr, X_va_raw, family_feats, k)

                X_tr_bal, y_tr_bal = make_balanced_subset(X_tr_sel, y_tr, 40, RANDOM_STATE)
                X_tr_q, X_va_q = prepare(X_tr_bal, y_tr_bal, X_va_sel, scale_quantum=True)

                fm = qmap_fn(k)
                qk = FidelityQuantumKernel(feature_map=fm)
                best_pred = 0; best_uar = -1
                for C in [0.1, 1, 10]:
                    m = QSVC(quantum_kernel=qk, C=C)
                    m.fit(X_tr_q, y_tr_bal)
                    p = m.predict(X_va_q)
                    u = recall_score([y[va_idx[0]]], p, average="macro")
                    if u > best_uar:
                        best_uar = u; best_pred = p[0]
                preds[va_idx] = best_pred

            m = compute_metrics(y, preds)
            search_results.append({
                "quantum_model": qname, "family": family_name, "k": k,
                "UAR": m["UAR"], "accuracy": m["accuracy"],
                "sensitivity": m["sensitivity"], "specificity": m["specificity"]
            })
            print(f"  {qname:25s} | {family_name:18s} k={k:2d}: "
                  f"UAR={m['UAR']:.4f} Acc={m['accuracy']:.4f}")

search_df = pd.DataFrame(search_results).sort_values("UAR", ascending=False)
print("\n===== TOP 10 QUANTUM CONFIGURATIONS =====")
display(search_df.head(10))

best_row    = search_df.iloc[0]
BEST_FAMILY = best_row["family"]
BEST_K      = int(best_row["k"])
BEST_QMODEL = best_row["quantum_model"]
BEST_FEATS  = FEATURE_FAMILIES[BEST_FAMILY]
print(f"\nBest: {BEST_QMODEL} | {BEST_FAMILY} k={BEST_K} → UAR={best_row['UAR']:.4f}")


## 4. Classical Baselines on Best Quantum Feature Set
Using the same feature family and top-K as the best quantum config.
Fair comparison: same features, same preprocessing, same LOO CV.
Classical baselines: Logistic Regression, Linear SVM, Random Forest.
(No heavily tuned ensembles — valid standard baselines.)


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier

CLASSICAL_MODELS = {
    "Logistic Regression": LogisticRegression(C=1, class_weight="balanced",
                               max_iter=2000, random_state=RANDOM_STATE),
    "SVM (Linear)"       : SVC(C=1, kernel="linear", class_weight="balanced",
                               random_state=RANDOM_STATE),
    "SVM (RBF)"          : SVC(C=10, kernel="rbf", gamma="scale",
                               class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest"      : RandomForestClassifier(n_estimators=200,
                               class_weight="balanced", random_state=RANDOM_STATE),
    "GradientBoosting"   : GradientBoostingClassifier(n_estimators=200,
                               max_depth=3, learning_rate=0.05,
                               random_state=RANDOM_STATE),
    "MLP"                : MLPClassifier(hidden_layer_sizes=(64, 32),
                               max_iter=500, early_stopping=True,
                               random_state=RANDOM_STATE),
}

X_best = df[BEST_FEATS].values
classical_results = []
loo = LeaveOneOut()

print(f"Classical baselines: {BEST_FAMILY}, top-{BEST_K} features")
print()

for model_name, model in CLASSICAL_MODELS.items():
    for use_smote in [False, True]:
        tag   = f"{model_name} + SMOTE" if use_smote else model_name
        preds = np.zeros(len(y), dtype=int)

        for tr_idx, va_idx in loo.split(X_best):
            X_tr_raw = X_best[tr_idx]; y_tr = y[tr_idx]
            X_va_raw = X_best[va_idx]

            X_tr_sel, X_va_sel, _ = select_top_k_inside_fold(
                X_tr_raw, y_tr, X_va_raw, BEST_FEATS, BEST_K)

            X_tr_s, X_va_s = prepare(X_tr_sel, y_tr, X_va_sel)

            if use_smote and y_tr.sum() > 1:
                sm = SMOTE(random_state=RANDOM_STATE,
                           k_neighbors=min(3, int(y_tr.sum())-1))
                X_tr_s, y_tr = sm.fit_resample(X_tr_s, y_tr)

            clf = clone(model)
            clf.fit(X_tr_s, y_tr)
            preds[va_idx] = clf.predict(X_va_s)

        m = compute_metrics(y, preds)
        classical_results.append({"model": tag, **m})
        print(f"  {tag:35s} UAR={m['UAR']:.4f} Acc={m['accuracy']:.4f} "
              f"Sen={m['sensitivity']:.4f} Spe={m['specificity']:.4f}")

classical_df = pd.DataFrame(classical_results).sort_values("UAR", ascending=False)
print(f"\nBest classical: {classical_df.iloc[0]['model']} "
      f"(UAR={classical_df.iloc[0]['UAR']:.4f})")


## 5. Matched SVM (same 40 samples as QSVC) — Isolates Quantum Kernel Effect

In [ ]:
print("Matched SVM (same 40 balanced samples as QSVC)...")
matched_preds = np.zeros(len(y), dtype=int)

for tr_idx, va_idx in loo.split(X_best):
    X_tr_raw = X_best[tr_idx]; y_tr = y[tr_idx]
    X_va_raw = X_best[va_idx]

    X_tr_sel, X_va_sel, _ = select_top_k_inside_fold(
        X_tr_raw, y_tr, X_va_raw, BEST_FEATS, BEST_K)

    # Same balanced subset as QSVC
    X_tr_bal, y_tr_bal = make_balanced_subset(X_tr_sel, y_tr, 40, RANDOM_STATE)
    X_tr_s, X_va_s     = prepare(X_tr_bal, y_tr_bal, X_va_sel)

    clf = SVC(C=1, kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE)
    clf.fit(X_tr_s, y_tr_bal)
    matched_preds[va_idx] = clf.predict(X_va_s)

matched_m = compute_metrics(y, matched_preds)
print(f"  Matched SVM: UAR={matched_m['UAR']:.4f} Acc={matched_m['accuracy']:.4f}")
print("  (Direct comparison to QSVC — isolates quantum kernel effect)")


## 6. Wilcoxon Significance Test

In [ ]:
from qiskit.circuit.library import ZFeatureMap

print("5-fold CV — Wilcoxon significance tests")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

QUANTUM_CONFIGS_TEST = [
    ("ZZFeatureMap reps=1", lambda d: ZZFeatureMap(feature_dimension=d, reps=1)),
    ("ZZFeatureMap reps=2", lambda d: ZZFeatureMap(feature_dimension=d, reps=2)),
    ("ZZFeatureMap reps=3", lambda d: ZZFeatureMap(feature_dimension=d, reps=3)),
    ("ZFeatureMap (linear)", lambda d: ZFeatureMap(feature_dimension=d, reps=2)),
]

fold_uars = {qname: [] for qname, _ in QUANTUM_CONFIGS_TEST}
fold_uars["SVM matched"]          = []
fold_uars["Logistic Regression"]  = []
fold_uars["SVM (RBF)"]            = []

X_best = df[BEST_FEATS].values

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_best, y), 1):
    X_tr_raw = X_best[tr_idx]; y_tr = y[tr_idx]
    X_va_raw = X_best[va_idx]; y_va = y[va_idx]

    X_tr_sel, X_va_sel, _ = select_top_k_inside_fold(
        X_tr_raw, y_tr, X_va_raw, BEST_FEATS, BEST_K)
    X_tr_bal, y_tr_bal = make_balanced_subset(X_tr_sel, y_tr, 40, RANDOM_STATE+fold)

    # All quantum models
    for qname, qmap_fn in QUANTUM_CONFIGS_TEST:
        X_tr_q, X_va_q = prepare(X_tr_bal, y_tr_bal, X_va_sel, scale_quantum=True)
        fm = qmap_fn(BEST_K)
        qk = FidelityQuantumKernel(feature_map=fm)
        best_m = None; best_u = -1
        for C in [0.1, 1, 10]:
            m = QSVC(quantum_kernel=qk, C=C)
            m.fit(X_tr_q, y_tr_bal)
            u = recall_score(y_va, m.predict(X_va_q), average="macro")
            if u > best_u: best_u = u; best_m = m
        fold_uars[qname].append(recall_score(y_va, best_m.predict(X_va_q), average="macro"))

    # Matched SVM (same 40 samples)
    X_tr_s, X_va_s = prepare(X_tr_bal, y_tr_bal, X_va_sel)
    sm = SVC(C=1, kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE)
    sm.fit(X_tr_s, y_tr_bal)
    fold_uars["SVM matched"].append(recall_score(y_va, sm.predict(X_va_s), average="macro"))

    # Classical on full training data
    X_tr_f, X_va_f = prepare(X_tr_sel, y_tr, X_va_sel)
    lr = LogisticRegression(C=1, class_weight="balanced", max_iter=2000,
                             random_state=RANDOM_STATE)
    lr.fit(X_tr_f, y_tr)
    fold_uars["Logistic Regression"].append(
        recall_score(y_va, lr.predict(X_va_f), average="macro"))

    rbf = SVC(C=10, kernel="rbf", gamma="scale", class_weight="balanced",
               random_state=RANDOM_STATE)
    rbf.fit(X_tr_f, y_tr)
    fold_uars["SVM (RBF)"].append(
        recall_score(y_va, rbf.predict(X_va_f), average="macro"))

    print(f"  Fold {fold}: " +
          " | ".join([f"{k[:8]}={np.mean(v):.3f}" for k, v in fold_uars.items()
                      if v]))

print(f"\n5-fold CV Summary:")
for name, uars in fold_uars.items():
    print(f"  {name:25s}: {np.mean(uars):.4f} ± {np.std(uars):.4f}")

# Wilcoxon tests
print("\n=== Wilcoxon Tests ===")

# Best quantum vs matched SVM
stat1, p1 = wilcoxon(fold_uars[BEST_QMODEL], fold_uars["SVM matched"],
                      alternative="two-sided")
print(f"Best QSVC ({BEST_QMODEL}) vs Matched SVM: p={p1:.4f}"
      + (" ✓ significant" if p1 < 0.05 else " (not significant)"))

# ZZFeatureMap vs ZFeatureMap — does entanglement help?
stat2, p2 = wilcoxon(fold_uars["ZZFeatureMap reps=2"],
                      fold_uars["ZFeatureMap (linear)"], alternative="two-sided")
print(f"ZZFeatureMap vs ZFeatureMap (entanglement): p={p2:.4f}"
      + (" ✓ significant" if p2 < 0.05 else " (not significant)"))

# reps=1 vs reps=2 vs reps=3 — circuit depth effect
stat3, p3 = wilcoxon(fold_uars["ZZFeatureMap reps=2"],
                      fold_uars["ZZFeatureMap reps=1"], alternative="two-sided")
print(f"reps=2 vs reps=1 (circuit depth): p={p3:.4f}"
      + (" ✓ significant" if p3 < 0.05 else " (not significant)"))

stat4, p4 = wilcoxon(fold_uars["ZZFeatureMap reps=2"],
                      fold_uars["ZZFeatureMap reps=3"], alternative="two-sided")
print(f"reps=2 vs reps=3 (circuit depth): p={p4:.4f}"
      + (" ✓ significant" if p4 < 0.05 else " (not significant)"))

p_val   = p1
p_val2  = p2


## 7. Final Results Summary

In [ ]:
print("=" * 70)
print("FINAL RESULTS — MODMA 128-Channel EEG")
print("=" * 70)
print(f"Feature config : {BEST_FAMILY}, top-{BEST_K}")
print(f"Best quantum   : {BEST_QMODEL}")
print(f"Validation     : Leave-One-Out CV (n=53)")
print()

rows = [{"Model": "Shi et al. 2020 (published, 3-ch)", "Type": "Baseline",
         "UAR": "—", "Accuracy": "72.25%",
         "Sensitivity": "74.06%", "Specificity": "63.63%"}]

for r in classical_df.to_dict("records"):
    rows.append({"Model": r["model"], "Type": "Classical",
                 "UAR": f"{r['UAR']:.4f}",
                 "Accuracy": f"{r['accuracy']*100:.2f}%",
                 "Sensitivity": f"{r['sensitivity']*100:.2f}%",
                 "Specificity": f"{r['specificity']*100:.2f}%"})

rows.append({"Model": "SVM (matched 40 samples)", "Type": "Classical (matched)",
             "UAR": f"{matched_m['UAR']:.4f}",
             "Accuracy": f"{matched_m['accuracy']*100:.2f}%",
             "Sensitivity": f"{matched_m['sensitivity']*100:.2f}%",
             "Specificity": f"{matched_m['specificity']*100:.2f}%"})

for _, r in search_df.iterrows():
    rows.append({"Model": f"QSVC {r['quantum_model']} | {r['family']} k={r['k']}",
                 "Type": "Quantum",
                 "UAR": f"{r['UAR']:.4f}",
                 "Accuracy": f"{r['accuracy']*100:.2f}%",
                 "Sensitivity": f"{r['sensitivity']*100:.2f}%",
                 "Specificity": f"{r['specificity']*100:.2f}%"})

summary_df = pd.DataFrame(rows)
display(summary_df)
summary_df.to_csv(BASE_SAVE / "modma_final_results.csv", index=False)

best_q = search_df.iloc[0]
with open(BASE_SAVE / "modma_quantum_summary.json", "w") as f:
    json.dump({
        "best_quantum_model"  : BEST_QMODEL,
        "best_family"         : BEST_FAMILY,
        "best_k"              : BEST_K,
        "best_qsvc_loo_uar"   : float(best_q["UAR"]),
        "best_qsvc_loo_acc"   : float(best_q["accuracy"]),
        "matched_svm_uar"     : float(matched_m["UAR"]),
        "wilcoxon_p_vs_matched"  : float(p_val),
        "wilcoxon_p_entanglement": float(p_val2),
        "published_baseline_acc" : 0.7225,
    }, f, indent=2)

print(f"\nBest QSVC    : UAR={best_q['UAR']:.4f} Acc={best_q['accuracy']*100:.2f}%")
print(f"Best classical: {classical_df.iloc[0]['model']} UAR={classical_df.iloc[0]['UAR']:.4f}")
print(f"Matched SVM  : UAR={matched_m['UAR']:.4f}")
print(f"\nWilcoxon QSVC vs matched: p={p_val:.4f}")
print(f"Wilcoxon entanglement   : p={p_val2:.4f}")
print("\nSaved to Google Drive.")
